# 🎓 Machine Learning Mini Project: Student Performance Clustering

**Course / Subject:** Machine Learning / Data Mining  
**Algorithm:** K-Means Clustering (Unsupervised Learning)  
**Features:** Attendance (%), Internal Marks (/100), Assignment Marks (/100), Study Hours (hrs/week)  
**Difficulty:** Easy  

---

## 📌 1. Problem Statement & Objectives
Identifying student academic trends early allows institutions to intervene proactively rather than reactively. In this notebook, we group students into distinct behavioral performance clusters without predefined labels, uncovering hidden patterns in attendance, exam performance, coursework completion, and independent study habits.

## 📦 2. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'font.size': 11})
print("All libraries imported successfully!")

## 📊 3. Dataset Loading & Inspection

In [ ]:
# Load dataset from local data folder
data_path = "../data/student_data.csv"
if not os.path.exists(data_path):
    data_path = "data/student_data.csv"

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape[0]} students, {df.shape[1]} columns\n")
display(df.head())
display(df.describe().round(2))

## 📈 4. Exploratory Data Analysis (EDA)
Let's visualize the distribution of each academic metric and their correlation.

In [ ]:
feature_cols = ["Attendance", "Internal_Marks", "Assignment_Marks", "Study_Hours"]

# Feature Distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, col in enumerate(feature_cols):
    ax = axes[i // 2, i % 2]
    sns.histplot(df[col], kde=True, ax=ax, color="#2b5c8f", bins=20)
    ax.set_title(f"Distribution of {col}", fontweight="bold")
plt.tight_layout()
plt.show()

# Correlation Heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df[feature_cols].corr(), annot=True, cmap="Blues", fmt=".2f", linewidths=1)
plt.title("Correlation Matrix of Academic Features", fontweight="bold")
plt.show()

## ⚖️ 5. Feature Scaling (StandardScaler)
**Why is scaling essential?**  
K-Means computes Euclidean distances between data points. Since `Study_Hours` spans $1-40$ while marks and attendance span $0-100$, unscaled features would allow marks to disproportionately dominate distance calculations. StandardScaler transforms all features to $\mu=0, \sigma=1$.

In [ ]:
X = df[feature_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled features sample (first 3 rows):")
print(X_scaled[:3].round(3))

## 🎯 6. Determining Optimal Clusters ($K$)
We evaluate two mathematical criteria across $K \in [1, 8]$:
1. **The Elbow Method (Inertia / WCSS):** Measures cluster compactness.
2. **Silhouette Score:** Evaluates cohesion vs separation (higher is better).

In [ ]:
k_range = range(1, 9)
inertia = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
    if k >= 2:
        sil_scores.append((k, silhouette_score(X_scaled, km.labels_)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Elbow plot
ax1.plot(list(k_range), inertia, marker='o', linewidth=2.5, color='#1f77b4')
ax1.axvline(x=3, color='#e74c3c', linestyle='--', label='Elbow Point (K=3)')
ax1.set_title("Elbow Method (WCSS vs K)", fontweight="bold")
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia (WCSS)")
ax1.legend()

# Silhouette plot
k_vals, s_vals = zip(*sil_scores)
ax2.bar(k_vals, s_vals, color='#2ca02c', alpha=0.85, width=0.5)
ax2.set_title("Silhouette Score by K", fontweight="bold")
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
for x, y in zip(k_vals, s_vals):
    ax2.text(x, y + 0.015, f"{y:.3f}", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 🚀 7. Training K-Means ($K=3$) & Profiling Performance Groups

In [ ]:
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=15)
df["Cluster_Raw"] = kmeans.fit_predict(X_scaled)

# Compute average metrics per cluster to dynamically label them
cluster_means = df.groupby("Cluster_Raw")[feature_cols].mean()
composite = cluster_means["Attendance"] + cluster_means["Internal_Marks"] + cluster_means["Assignment_Marks"]
sorted_ids = composite.sort_values().index.tolist()

label_map = {
    sorted_ids[0]: "At-Risk / Needs Intervention",
    sorted_ids[1]: "Moderate / Consistent Performers",
    sorted_ids[2]: "High Achievers / Distinction"
}

df["Performance_Group"] = df["Cluster_Raw"].map(label_map)

summary = df.groupby("Performance_Group")[feature_cols].mean().round(1)
summary["Student_Count"] = df.groupby("Performance_Group")["Student_ID"].count()
display(summary)

## 🗺️ 8. Dimensionality Reduction (PCA 2D Projection)
We reduce the 4 features into 2 Principal Components to visually examine cluster boundaries and centroids.

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)
df["PC1"] = pca_coords[:, 0]
df["PC2"] = pca_coords[:, 1]

exp_var = pca.explained_variance_ratio_ * 100
print(f"Total Variance Explained by 2 Components: {sum(exp_var):.2f}%\n")

plt.figure(figsize=(10, 6))
colors = {"At-Risk / Needs Intervention": "#e74c3c", "Moderate / Consistent Performers": "#f39c12", "High Achievers / Distinction": "#27ae60"}

for group, color in colors.items():
    sub = df[df["Performance_Group"] == group]
    plt.scatter(sub["PC1"], sub["PC2"], label=f"{group} (n={len(sub)})", color=color, alpha=0.75, edgecolors='k', linewidth=0.5, s=60)

# Plot Centroids
centroids_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centroids_pca[:, 0], centroids_pca[:, 1], marker='X', color='black', s=200, edgecolors='white', linewidth=2, label='Cluster Centroids', zorder=10)

plt.title("Student Performance Clusters (2D PCA Space)", fontsize=14, fontweight="bold")
plt.xlabel(f"Principal Component 1 ({exp_var[0]:.1f}% Variance)")
plt.ylabel(f"Principal Component 2 ({exp_var[1]:.1f}% Variance)")
plt.legend(frameon=True, facecolor='white')
plt.tight_layout()
plt.show()

## 🏆 9. Algorithm Comparison (K-Means vs Agglomerative vs GMM)
We benchmark K-Means against other major clustering paradigms using Silhouette, Davies-Bouldin, and Calinski-Harabasz metrics.

In [ ]:
algorithms = {
    "K-Means": KMeans(n_clusters=3, random_state=42, n_init=10),
    "Agglomerative (Hierarchical)": AgglomerativeClustering(n_clusters=3, linkage='ward'),
    "Gaussian Mixture (GMM)": GaussianMixture(n_components=3, random_state=42)
}

benchmark = []
for name, model in algorithms.items():
    lbls = model.fit_predict(X_scaled)
    benchmark.append({
        "Algorithm": name,
        "Silhouette Score (Higher Better)": round(silhouette_score(X_scaled, lbls), 4),
        "Davies-Bouldin Index (Lower Better)": round(davies_bouldin_score(X_scaled, lbls), 4),
        "Calinski-Harabasz Score (Higher Better)": round(calinski_harabasz_score(X_scaled, lbls), 2)
    })

display(pd.DataFrame(benchmark))

## 🔮 10. Real-Time Student Prediction Function
Let's test classifying a new student who was not part of the training dataset.

In [ ]:
def classify_new_student(attendance, internal_marks, assignment_marks, study_hours):
    new_data = pd.DataFrame([[attendance, internal_marks, assignment_marks, study_hours]], columns=feature_cols)
    new_scaled = scaler.transform(new_data)
    cluster_id = kmeans.predict(new_scaled)[0]
    group_name = label_map[cluster_id]
    
    return {
        "Assigned Group": group_name,
        "Attendance": f"{attendance}%",
        "Internal Marks": f"{internal_marks}/100",
        "Assignment Marks": f"{assignment_marks}/100",
        "Weekly Study Hours": f"{study_hours} hrs"
    }

# Test student 1: High engagement
print("Student 1:", classify_new_student(attendance=92.0, internal_marks=88.0, assignment_marks=91.0, study_hours=28.0))

# Test student 2: Needs attention
print("Student 2:", classify_new_student(attendance=50.0, internal_marks=40.0, assignment_marks=42.0, study_hours=5.0))

## 🎯 11. Conclusion & Recommendations
1. **Clear Behavioral Segments:** Attendance and self-study hours serve as strong behavioral precursors to exam and assignment performance.
2. **Early Intervention:** Rather than waiting for semester-end results, institutions can flag 'At-Risk' students during midterm evaluations.
3. **Scalability:** K-Means enables near-instantaneous inference for thousands of students with low computational complexity.